# Synthetic Data Generation → Object Detection Training

Fine-tune an object detector on **synthetic data generated with NVIDIA Omniverse Replicator**,
training natively on this machine's GPU with **PyTorch** (Faster R-CNN, ResNet-18 + FPN backbone).

| Step | What happens |
|---|---|
| 0 | Prerequisites check — GPU, CUDA, compute capability |
| 1 | Configuration — paths and hyperparameters |
| 2 | Set up the training environment |
| 3 | Load a pre-trained object detection backbone |
| 4 | Convert the dataset: Replicator → KITTI → `torch.utils.data.Dataset` |
| 5 | Specify training parameters (batch size, learning rate, …) |
| 6 | Train the model |
| 7 | Evaluate on held-out test data (COCO-style mAP) |
| 8 | Visualize results |
| 9 | *Optional* — export to TorchScript / ONNX |

At the end you will have either a fine-tuned detector of your own, or you can fall back to the
pre-trained model shipped with the module.

---

### Why not TAO Toolkit?

The reference workflow uses TAO Toolkit's `detectnet_v2`. That ships only in a **linux/amd64**
container, and this host is **aarch64** (GB10 / DGX Spark). This is not a performance limitation
that emulation can work around: `nvidia-container-toolkit` injects the *host's* aarch64
`libcuda.so` into the container, and an emulated x86 process cannot load an aarch64 shared object.
Emulation would give you an amd64 userspace with no CUDA at all.

So this notebook trains the same thing — a ResNet-18-backboned detector, from an ImageNet-pretrained
backbone, on Replicator-generated synthetic data — using a native arm64 PyTorch stack instead.
The GPU here is a Blackwell part and is comfortably faster than the RTX A6000 the original
"~1 hour" figure refers to.

> If you need TAO specifically (e.g. to export a DeepStream `.etlt`), run it on an x86_64 host and
> copy the result back. Steps 4 and 8 here are framework-agnostic and still apply.

---
## 0. Prerequisites check

Confirms the GPU is visible to PyTorch and that the installed build actually has kernels for this
device — the failure mode on very new hardware is a CUDA-capable install that has no matching
`sm_` binary and dies at the first kernel launch.

In [ ]:
import json
import os
import platform
import random
import shutil
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import torch
import torchvision

print(f"Python      : {sys.version.split()[0]}")
print(f"Platform    : {platform.system()} {platform.machine()}")
print(f"torch       : {torch.__version__}")
print(f"torchvision : {torchvision.__version__}")
print(f"CUDA avail  : {torch.cuda.is_available()}")

if not torch.cuda.is_available():
    print(
        "\nNo CUDA device. Install a matching build, e.g.:\n"
        "  pip install --index-url https://download.pytorch.org/whl/cu130 torch torchvision"
    )
else:
    cap = torch.cuda.get_device_capability(0)
    arches = torch.cuda.get_arch_list()
    props = torch.cuda.get_device_properties(0)
    print(f"device      : {props.name}")
    print(f"capability  : sm_{cap[0]}{cap[1]}")
    print(f"vram        : {props.total_memory / 1e9:.0f} GB")
    print(f"build archs : {arches}")

    # Prove a real kernel launches, rather than trusting is_available().
    try:
        _m = torch.nn.Conv2d(3, 16, 3, padding=1).cuda()
        _y = _m(torch.randn(2, 3, 64, 64, device="cuda")).sum()
        _y.backward()
        torch.cuda.synchronize()
        print("kernel test : OK (conv2d forward + backward)")
    except Exception as exc:  # noqa: BLE001
        print(f"kernel test : FAILED -> {exc}")
        print("  Your torch build has no kernel for this GPU. Install a newer build.")

    if f"sm_{cap[0]}{cap[1]}" not in arches:
        print(
            f"\nNote: no exact sm_{cap[0]}{cap[1]} binary in this build; CUDA is falling back to a\n"
            "compatible arch from the list above. That is fine as long as the kernel test passed."
        )

---
## 1. Configuration

Everything downstream reads from this cell. Copy `.env.example` to `.env` to override any of it
without editing the notebook.

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

def env(key, default):
    return os.environ.get(key) or default

# --- Paths --------------------------------------------------------------------
PROJECT_DIR = Path(env("LOCAL_PROJECT_DIR", Path.cwd().parent)).resolve()
RAW_DATA_DIR = PROJECT_DIR / env("RAW_DATA_DIR", "data/raw")      # Replicator output
KITTI_DIR    = PROJECT_DIR / env("KITTI_DATA_DIR", "data/kitti")  # converted
RESULTS_DIR  = PROJECT_DIR / "results"
for d in (RAW_DATA_DIR, KITTI_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Dataset ------------------------------------------------------------------
TARGET_CLASS = env("TARGET_CLASS", "palletjack")
VAL_SPLIT    = float(env("VAL_SPLIT", "0.2"))
RANDOM_SEED  = int(env("RANDOM_SEED", "42"))
IMAGE_WIDTH  = int(env("IMAGE_WIDTH", "1280"))
IMAGE_HEIGHT = int(env("IMAGE_HEIGHT", "720"))

# Class 0 is reserved for background by torchvision detection models.
CLASSES = ["__background__", TARGET_CLASS]
NUM_CLASSES = len(CLASSES)

# --- Model / training ---------------------------------------------------------
BACKBONE      = env("BACKBONE", "resnet18")
BATCH_SIZE    = int(env("BATCH_SIZE", "16"))
NUM_EPOCHS    = int(env("NUM_EPOCHS", "20"))
LEARNING_RATE = float(env("LEARNING_RATE", "2e-2"))
WEIGHT_DECAY  = float(env("WEIGHT_DECAY", "5e-4"))
NUM_WORKERS   = int(env("NUM_WORKERS", "8"))
USE_AMP       = env("USE_AMP", "1") == "1"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

assert 0.0 < VAL_SPLIT < 1.0, "VAL_SPLIT must be a fraction strictly between 0 and 1"

print(f"Project dir  : {PROJECT_DIR}")
print(f"Raw data     : {RAW_DATA_DIR}")
print(f"KITTI data   : {KITTI_DIR}")
print(f"Classes      : {CLASSES}")
print(f"Image size   : {IMAGE_WIDTH}x{IMAGE_HEIGHT}")
print(f"Backbone     : {BACKBONE}")
print(f"Batch/epochs : {BATCH_SIZE} / {NUM_EPOCHS} @ lr={LEARNING_RATE}")
print(f"Device       : {DEVICE}  | AMP: {USE_AMP}")

---
## 2. Set up the training environment

No container needed — training runs in this process on the local GPU. This cell enables the
throughput knobs that matter on Blackwell (TF32 matmuls, cuDNN autotuning) and measures what the
device actually delivers, so the epoch-time estimate in Step 6 is grounded in a real number.

In [ ]:
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True          # autotune convs for fixed input sizes
    torch.cuda.empty_cache()

    def _bench(dtype, n=2048, iters=30):
        a = torch.randn(n, n, device="cuda", dtype=dtype)
        for _ in range(5):
            a @ a
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        for _ in range(iters):
            a @ a
        torch.cuda.synchronize()
        secs = time.perf_counter() - t0
        return iters * 2 * n ** 3 / secs / 1e12

    print(f"matmul fp32/tf32 : {_bench(torch.float32):5.1f} TFLOP/s")
    print(f"matmul bf16      : {_bench(torch.bfloat16):5.1f} TFLOP/s")
    print(f"\nTF32 + cuDNN autotune enabled. AMP={'on' if USE_AMP else 'off'}.")
else:
    print("Running on CPU — training will be extremely slow. Fix Step 0 first.")

---
## 3. Load a pre-trained object detection model

Starting from an ImageNet-pretrained **ResNet-18** rather than random weights is the difference
between a detector that works on a few thousand synthetic images and one that does not.

`resnet_fpn_backbone` wraps the classifier in a Feature Pyramid Network so the detector sees
multiple scales; `trainable_layers=3` freezes the earliest (most generic) blocks, which both
speeds up training and reduces overfitting on synthetic data.

In [ ]:
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.models.detection.rpn import AnchorGenerator

def build_model(num_classes, backbone_name="resnet18", trainable_layers=3, pretrained=True):
    """Faster R-CNN with an ImageNet-pretrained ResNet + FPN backbone."""
    weights = "IMAGENET1K_V1" if pretrained else None
    backbone = resnet_fpn_backbone(
        backbone_name=backbone_name,
        weights=weights,
        trainable_layers=trainable_layers,
    )
    # One anchor size per FPN level (P2..P5 + pool = 5 levels).
    anchor_gen = AnchorGenerator(
        sizes=((32,), (64,), (128,), (256,), (512,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5,
    )
    return FasterRCNN(
        backbone,
        num_classes=num_classes,
        rpn_anchor_generator=anchor_gen,
        min_size=IMAGE_HEIGHT,
        max_size=IMAGE_WIDTH,
    )

model = build_model(NUM_CLASSES, BACKBONE).to(DEVICE)

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"model      : FasterRCNN + {BACKBONE}-FPN")
print(f"classes    : {NUM_CLASSES} {CLASSES}")
print(f"parameters : {n_total/1e6:.1f} M total, {n_train/1e6:.1f} M trainable")

---
## 4. Prepare the dataset

Two hops: **Replicator output → KITTI → `Dataset`**. KITTI is kept as the on-disk intermediate
because it is human-inspectable and is what the rest of the ecosystem (including TAO) expects, so
the converted data stays portable.

Expected Replicator (`BasicWriter`) layout under `data/raw/`:

```
data/raw/<any-subdir>/
├── rgb_0000.png
├── bounding_box_2d_tight_0000.npy
├── bounding_box_2d_tight_labels_0000.json
└── ...
```

In [ ]:
# 4.1 — Replicator -> KITTI converter.
from PIL import Image

# KITTI label columns: type truncated occluded alpha x1 y1 x2 y2 h w l x y z ry
KITTI_LINE = "{cls} 0.00 0 0.00 {x1:.2f} {y1:.2f} {x2:.2f} {y2:.2f} 0.00 0.00 0.00 0.00 0.00 0.00 0.00"

def _boxes_from_npy(npy_path):
    """Return [(semantic_id, x1, y1, x2, y2, occlusion), ...] from a Replicator bbox file."""
    arr = np.load(str(npy_path), allow_pickle=True)
    out = []
    names = arr.dtype.names
    for row in arr:
        if names:  # structured array — the normal case
            sid = int(row["semanticId"])
            x1, y1, x2, y2 = (float(row[k]) for k in ("x_min", "y_min", "x_max", "y_max"))
            occ = float(row["occlusionRatio"]) if "occlusionRatio" in names else 0.0
        else:      # plain Nx5/Nx6 array
            sid, x1, y1, x2, y2 = (float(v) for v in row[:5])
            occ = float(row[5]) if len(row) > 5 else 0.0
        out.append((int(sid), x1, y1, x2, y2, occ))
    return out

def convert_replicator_to_kitti(
    raw_dir, out_dir, target_class,
    max_occlusion=0.9, min_box_px=8, image_size=None,
):
    """Convert Replicator BasicWriter output to a flat KITTI dataset.

    Boxes are scaled to `image_size`, clamped to the frame, then dropped if they are
    more than `max_occlusion` occluded, degenerate (zero area after clamping), or
    smaller than `min_box_px` on either side.

    `min_box_px` is measured in **output** pixels, i.e. after resizing. Note the
    consequence when upscaling: at 640->1280 a 4 px source box becomes 8 px and
    survives, despite carrying no real signal. Set it relative to your output
    resolution, not your source.

    Returns a stats dict.
    """
    raw_dir, out_dir = Path(raw_dir), Path(out_dir)
    img_out, lbl_out = out_dir / "images", out_dir / "labels"
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    rgb_files = sorted(raw_dir.rglob("rgb_*.png")) + sorted(raw_dir.rglob("rgb_*.jpg"))
    stats = {"frames": 0, "empty_frames": 0, "boxes_kept": 0, "boxes_dropped": 0, "missing": 0}

    for idx, rgb in enumerate(rgb_files):
        suffix = rgb.stem.split("_")[-1]                      # "0000"
        npy = rgb.parent / f"bounding_box_2d_tight_{suffix}.npy"
        js  = rgb.parent / f"bounding_box_2d_tight_labels_{suffix}.json"
        if not npy.exists():
            stats["missing"] += 1
            continue

        id2class = {}
        if js.exists():
            for k, v in json.loads(js.read_text()).items():
                id2class[int(k)] = (v.get("class") if isinstance(v, dict) else str(v)).lower()

        with Image.open(rgb) as im:
            W, H = im.size
            stem = f"frame_{idx:06d}"
            if image_size and (W, H) != tuple(image_size):
                im = im.convert("RGB").resize(tuple(image_size), Image.BILINEAR)
                sx, sy = image_size[0] / W, image_size[1] / H
                W, H = image_size
            else:
                im = im.convert("RGB")
                sx = sy = 1.0
            im.save(img_out / f"{stem}.png")

        lines = []
        for sid, x1, y1, x2, y2, occ in _boxes_from_npy(npy):
            name = id2class.get(sid, target_class)
            if name != target_class:
                stats["boxes_dropped"] += 1
                continue
            x1, x2 = sorted((x1 * sx, x2 * sx))
            y1, y2 = sorted((y1 * sy, y2 * sy))
            x1, y1 = max(0.0, x1), max(0.0, y1)
            x2, y2 = min(float(W - 1), x2), min(float(H - 1), y2)
            w, h = x2 - x1, y2 - y1
            if occ > max_occlusion or w <= 0 or h <= 0 or w < min_box_px or h < min_box_px:
                stats["boxes_dropped"] += 1
                continue
            lines.append(KITTI_LINE.format(cls=target_class, x1=x1, y1=y1, x2=x2, y2=y2))
            stats["boxes_kept"] += 1

        (lbl_out / f"{stem}.txt").write_text("\n".join(lines) + ("\n" if lines else ""))
        stats["frames"] += 1
        if not lines:
            stats["empty_frames"] += 1

    return stats

print("converter defined")

In [ ]:
# 4.2 — Run the conversion.
if not list(RAW_DATA_DIR.rglob("rgb_*.png")):
    print(
        f"No Replicator frames under {RAW_DATA_DIR}.\n"
        "Generate them with Omniverse Replicator, or drop an existing KITTI dataset\n"
        f"into {KITTI_DIR}/{{images,labels}} and skip to 4.3."
    )
else:
    stats = convert_replicator_to_kitti(
        RAW_DATA_DIR, KITTI_DIR, TARGET_CLASS, image_size=(IMAGE_WIDTH, IMAGE_HEIGHT),
    )
    print(json.dumps(stats, indent=2))
    if stats["frames"] and stats["empty_frames"] / stats["frames"] > 0.5:
        print(
            "\nWARNING: over half the frames have no boxes. Check that TARGET_CLASS matches"
            " the semantic label used in your Replicator scene."
        )

In [ ]:
# 4.3 — Sanity check: counts, box statistics, class balance.
images = sorted((KITTI_DIR / "images").glob("*.png"))
labels = sorted((KITTI_DIR / "labels").glob("*.txt"))
assert images, f"no images in {KITTI_DIR / 'images'}"
assert len(images) == len(labels), f"{len(images)} images vs {len(labels)} labels — mismatch"

box_count, widths, heights, classes = 0, [], [], {}
for lbl in labels:
    for line in lbl.read_text().splitlines():
        p = line.split()
        if len(p) < 8:
            continue
        classes[p[0]] = classes.get(p[0], 0) + 1
        x1, y1, x2, y2 = (float(v) for v in p[4:8])
        widths.append(x2 - x1); heights.append(y2 - y1)
        box_count += 1

print(f"images  : {len(images)}")
print(f"boxes   : {box_count}  ({box_count / max(len(images), 1):.2f} per image)")
print(f"classes : {classes}")
if widths:
    print(f"box w px: min {min(widths):.0f} / median {sorted(widths)[len(widths)//2]:.0f} / max {max(widths):.0f}")
    print(f"box h px: min {min(heights):.0f} / median {sorted(heights)[len(heights)//2]:.0f} / max {max(heights):.0f}")
unknown = set(classes) - set(CLASSES)
if unknown:
    print(f"\nWARNING: labels contain classes not in CLASSES and will be ignored: {unknown}")

In [ ]:
# 4.4 — Dataset + train/val split.
from torch.utils.data import DataLoader, Dataset

class KittiDetectionDataset(Dataset):
    """KITTI-layout detection dataset returning torchvision-style (image, target)."""

    def __init__(self, root, classes, stems=None, hflip_prob=0.0, jitter=None):
        self.root = Path(root)
        self.classes = list(classes)
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}
        self.hflip_prob = hflip_prob
        self.jitter = jitter
        if stems is None:
            stems = sorted(p.stem for p in (self.root / "images").glob("*.png"))
        self.stems = list(stems)

    def __len__(self):
        return len(self.stems)

    def _load_boxes(self, stem):
        boxes, labels = [], []
        lbl = self.root / "labels" / f"{stem}.txt"
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                p = line.split()
                if len(p) < 8 or p[0] not in self.class_to_idx:
                    continue
                x1, y1, x2, y2 = (float(v) for v in p[4:8])
                if x2 <= x1 or y2 <= y1:
                    continue
                boxes.append([x1, y1, x2, y2])
                labels.append(self.class_to_idx[p[0]])
        return boxes, labels

    def __getitem__(self, i):
        stem = self.stems[i]
        img = Image.open(self.root / "images" / f"{stem}.png").convert("RGB")
        boxes, labels = self._load_boxes(stem)

        if self.jitter is not None:
            img = self.jitter(img)

        img_t = torchvision.transforms.functional.to_tensor(img)
        boxes_t = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels_t = torch.as_tensor(labels, dtype=torch.int64)

        if self.hflip_prob and random.random() < self.hflip_prob:
            img_t = img_t.flip(-1)
            if len(boxes_t):
                W = img_t.shape[-1]
                boxes_t = boxes_t[:, [2, 1, 0, 3]] * torch.tensor([-1.0, 1, -1, 1]) \
                          + torch.tensor([W - 1.0, 0, W - 1.0, 0])

        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([i]),
            "area": (boxes_t[:, 3] - boxes_t[:, 1]) * (boxes_t[:, 2] - boxes_t[:, 0])
                    if len(boxes_t) else torch.zeros(0),
            "iscrowd": torch.zeros(len(boxes_t), dtype=torch.int64),
        }
        return img_t, target

def collate_fn(batch):
    return tuple(zip(*batch))

# Deterministic split, held fixed by RANDOM_SEED.
all_stems = sorted(p.stem for p in (KITTI_DIR / "images").glob("*.png"))
rng = random.Random(RANDOM_SEED)
shuffled = all_stems[:]
rng.shuffle(shuffled)
n_val = max(1, int(len(shuffled) * VAL_SPLIT))
val_stems, train_stems = shuffled[:n_val], shuffled[n_val:]

# Synthetic data has no sensor noise; colour jitter is most of what closes the sim-to-real gap.
jitter = torchvision.transforms.ColorJitter(
    brightness=0.3, contrast=0.3, saturation=0.3, hue=0.05
)
train_ds = KittiDetectionDataset(KITTI_DIR, CLASSES, train_stems, hflip_prob=0.5, jitter=jitter)
val_ds   = KittiDetectionDataset(KITTI_DIR, CLASSES, val_stems)

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    collate_fn=collate_fn, pin_memory=(DEVICE.type == "cuda"), persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    collate_fn=collate_fn, pin_memory=(DEVICE.type == "cuda"), persistent_workers=NUM_WORKERS > 0,
)

print(f"train : {len(train_ds)} images, {len(train_loader)} batches")
print(f"val   : {len(val_ds)} images, {len(val_loader)} batches")
_img, _tgt = train_ds[0]
print(f"sample: image {tuple(_img.shape)} {_img.dtype}, boxes {tuple(_tgt['boxes'].shape)},"
      f" labels {_tgt['labels'].tolist()}")

In [ ]:
# 4.5 — Eyeball the ground truth before training on it.
import matplotlib.patches as patches
import matplotlib.pyplot as plt

def show_samples(ds, n=6, title="Ground truth", seed=RANDOM_SEED):
    idxs = random.Random(seed).sample(range(len(ds)), min(n, len(ds)))
    cols = 3
    rows = (len(idxs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 3 * rows))
    for ax, i in zip(np.array(axes).ravel(), idxs):
        img, tgt = ds[i]
        ax.imshow(img.permute(1, 2, 0).numpy())
        for box in tgt["boxes"]:
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(patches.Rectangle(
                (x1, y1), x2 - x1, y2 - y1, linewidth=2, edgecolor="lime", facecolor="none"))
        ax.set_title(f"{ds.stems[i]} — {len(tgt['boxes'])} box(es)", fontsize=9)
        ax.axis("off")
    for ax in np.array(axes).ravel()[len(idxs):]:
        ax.axis("off")
    fig.suptitle(title, fontsize=13)
    fig.tight_layout()
    plt.show()

show_samples(train_ds, title="Ground truth (train, with augmentation)")

---
## 5. Specify training parameters

| Knob | Effect |
|---|---|
| `BATCH_SIZE` | See the note below — the default is tuned for this machine |
| `NUM_EPOCHS` | Main wall-clock lever |
| `LEARNING_RATE` | Scale roughly linearly with batch size |
| `trainable_layers` | How much of the backbone unfreezes (Step 3) |
| `USE_AMP` | Mixed precision — large speedup on Blackwell, minimal accuracy cost |

SGD with momentum and a cosine schedule is the standard recipe for Faster R-CNN; a short warmup
avoids the loss spike that an untrained detection head otherwise causes in the first few hundred
iterations.

**Batch size on this machine.** Measured on the GB10 at 1280x720 with this exact model, AMP on:

| batch | peak GPU memory | throughput |
|---|---|---|
| 4  | 2.2 GB  |  9.7 img/s |
| 8  | 4.0 GB  | 10.5 img/s |
| 16 | 7.5 GB  | 10.7 img/s |
| 32 | 14.6 GB | 10.7 img/s |

Throughput plateaus at 16, and even 32 uses barely a tenth of the 130 GB unified memory — so the
usual "raise it until you OOM" advice is the wrong instinct here. You will never OOM before you
stop gaining speed. The default is 16.

`LEARNING_RATE` is scaled to match: the standard Faster R-CNN recipe is `lr = 0.02` at batch 16,
so **if you change `BATCH_SIZE`, scale the LR linearly** (batch 8 -> 1e-2, batch 32 -> 4e-2).

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=LEARNING_RATE, momentum=0.9, weight_decay=WEIGHT_DECAY)

iters_per_epoch = max(1, len(train_loader))
total_iters = NUM_EPOCHS * iters_per_epoch
warmup_iters = min(500, total_iters // 10)

def lr_at(it):
    """Linear warmup then cosine decay, as a multiplier on the base LR."""
    if it < warmup_iters:
        return (it + 1) / max(1, warmup_iters)
    progress = (it - warmup_iters) / max(1, total_iters - warmup_iters)
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_at)
scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and DEVICE.type == "cuda"))

print(f"optimizer   : SGD(lr={LEARNING_RATE}, momentum=0.9, wd={WEIGHT_DECAY})")
print(f"schedule    : {warmup_iters} warmup iters -> cosine over {total_iters} total")
print(f"iters/epoch : {iters_per_epoch}")
print(f"AMP         : {scaler.is_enabled()}")

---
## 6. Train the model

The first epoch prints a measured throughput figure and an estimate for the full run. Checkpoints
go to `results/`, and the best model by validation mAP is kept separately — detection models
overfit synthetic data readily, so the last epoch is often not the best one.

In [ ]:
def train_one_epoch(model, loader, optimizer, scheduler, scaler, epoch, log_every=20):
    model.train()
    running, seen, t0 = {}, 0, time.perf_counter()
    for i, (images, targets) in enumerate(loader):
        images = [img.to(DEVICE, non_blocking=True) for img in images]
        targets = [{k: v.to(DEVICE, non_blocking=True) for k, v in t.items()} for t in targets]

        with torch.amp.autocast("cuda", enabled=scaler.is_enabled()):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

        if not torch.isfinite(loss):
            print(f"  [!] non-finite loss at iter {i}: {loss.item()} — skipping batch")
            optimizer.zero_grad(set_to_none=True)
            continue

        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        for k, v in loss_dict.items():
            running[k] = running.get(k, 0.0) + v.item()
        running["total"] = running.get("total", 0.0) + loss.item()
        seen += 1

        if (i + 1) % log_every == 0 or i + 1 == len(loader):
            imgs_s = (i + 1) * loader.batch_size / (time.perf_counter() - t0)
            parts = " ".join(f"{k}={running[k]/seen:.3f}" for k in sorted(running))
            print(f"  epoch {epoch} [{i+1:>4}/{len(loader)}] {parts} "
                  f"lr={optimizer.param_groups[0]['lr']:.2e} {imgs_s:.1f} img/s")

    return {k: v / max(seen, 1) for k, v in running.items()}, time.perf_counter() - t0

In [ ]:
history = []
best_map, best_path = -1.0, RESULTS_DIR / "model_best.pt"
last_path = RESULTS_DIR / "model_last.pt"

for epoch in range(1, NUM_EPOCHS + 1):
    losses, secs = train_one_epoch(model, train_loader, optimizer, scheduler, scaler, epoch)
    print(f"epoch {epoch}/{NUM_EPOCHS} done in {secs:.0f}s — loss {losses.get('total', 0):.4f}")
    if epoch == 1:
        print(f"  -> estimated total training time: {secs * NUM_EPOCHS / 60:.0f} min")

    torch.save({"epoch": epoch, "model": model.state_dict(),
                "classes": CLASSES, "backbone": BACKBONE}, last_path)
    history.append({"epoch": epoch, **losses, "secs": secs})

print(f"\ntraining complete — last checkpoint: {last_path}")

In [ ]:
# Loss curve.
if history:
    fig, ax = plt.subplots(figsize=(9, 4))
    epochs = [h["epoch"] for h in history]
    for k in sorted({k for h in history for k in h} - {"epoch", "secs"}):
        ax.plot(epochs, [h.get(k, np.nan) for h in history], marker="o", ms=3, label=k)
    ax.set_xlabel("epoch"); ax.set_ylabel("loss"); ax.set_yscale("log")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_title("Training loss")
    plt.tight_layout(); plt.show()

---
## 7. Evaluate on held-out test data

COCO-style average precision, computed directly rather than via `pycocotools` so there is nothing
to compile on aarch64. Predictions are greedily matched to ground truth in descending score order,
each ground-truth box claimed at most once; AP is the area under the resulting
precision-recall curve (all-point interpolation).

**Reading the result:** AP\@0.5 above ~0.85 on *synthetic* validation is expected and easy. The
number that matters is AP on *real* images, which is typically much lower. That gap is the
sim-to-real gap, and closing it is what more aggressive domain randomization in Replicator is for.

In [ ]:
def box_iou_matrix(a, b):
    """IoU between boxes a [N,4] and b [M,4] in xyxy -> [N,M]."""
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    area_a = np.prod(np.clip(a[:, 2:] - a[:, :2], 0, None), axis=1)
    area_b = np.prod(np.clip(b[:, 2:] - b[:, :2], 0, None), axis=1)
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / np.maximum(union, 1e-12), 0.0)

def average_precision(records, n_gt, iou_thr):
    """AP for one class via all-point interpolation.

    `records` is {"preds": [(score, image_id, box), ...], "gts": {image_id: boxes}}.
    Returns NaN when the class has no ground truth (undefined, not zero) so it can be
    excluded from the mean rather than dragging it down.
    """
    if n_gt == 0:
        return float("nan")
    preds, gts = records["preds"], records["gts"]
    if not preds:
        return 0.0
    order = np.argsort([-p[0] for p in preds])
    matched = {img_id: np.zeros(len(boxes), dtype=bool) for img_id, boxes in gts.items()}
    tp = np.zeros(len(order)); fp = np.zeros(len(order))

    for rank, pi in enumerate(order):
        _, img_id, box = preds[pi]
        gt_boxes = gts.get(img_id, np.zeros((0, 4)))
        if len(gt_boxes) == 0:
            fp[rank] = 1
            continue
        ious = box_iou_matrix(np.asarray([box]), gt_boxes)[0]
        best = int(np.argmax(ious))
        if ious[best] >= iou_thr and not matched[img_id][best]:
            matched[img_id][best] = True
            tp[rank] = 1
        else:
            fp[rank] = 1

    tp_c, fp_c = np.cumsum(tp), np.cumsum(fp)
    recall = tp_c / n_gt
    precision = tp_c / np.maximum(tp_c + fp_c, 1e-12)
    # Monotonically decreasing precision envelope, then integrate.
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    idx = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[idx + 1] - mrec[idx]) * mpre[idx + 1]))

@torch.no_grad()
def evaluate(model, loader, classes, iou_thrs=None, score_thr=0.05):
    """Return {class: {AP50, AP75, mAP}} plus overall mAP@[.5:.95]."""
    if iou_thrs is None:
        iou_thrs = np.arange(0.5, 1.0, 0.05)
    model.eval()
    per_class = {c: {"preds": [], "gts": {}} for c in classes[1:]}
    n_gt = {c: 0 for c in classes[1:]}

    img_id = 0
    for images, targets in loader:
        images = [img.to(DEVICE, non_blocking=True) for img in images]
        outputs = model(images)
        for out, tgt in zip(outputs, targets):
            gt_b = tgt["boxes"].cpu().numpy(); gt_l = tgt["labels"].cpu().numpy()
            for ci, cname in enumerate(classes[1:], start=1):
                sel = gt_b[gt_l == ci]
                per_class[cname]["gts"][img_id] = sel
                n_gt[cname] += len(sel)
            pb = out["boxes"].cpu().numpy()
            ps = out["scores"].cpu().numpy()
            pl = out["labels"].cpu().numpy()
            keep = ps >= score_thr
            for b, s, l in zip(pb[keep], ps[keep], pl[keep]):
                if 1 <= l < len(classes):
                    per_class[classes[l]]["preds"].append((float(s), img_id, b))
            img_id += 1

    results, all_maps = {}, []
    for cname in classes[1:]:
        aps = [average_precision(per_class[cname], n_gt[cname], t) for t in iou_thrs]
        aps = np.array(aps, dtype=float)
        results[cname] = {
            "AP50": aps[0],
            "AP75": aps[int(np.argmin(np.abs(iou_thrs - 0.75)))],
            "mAP": float(np.nanmean(aps)),
            "n_gt": n_gt[cname],
        }
        all_maps.append(results[cname]["mAP"])
    return results, float(np.nanmean(all_maps)) if all_maps else float("nan")

print("evaluation defined")

In [ ]:
results, overall_map = evaluate(model, val_loader, CLASSES)

print(f"Validation — {len(val_ds)} images")
print("-" * 58)
print(f"{'class':<18}{'AP@0.5':>9}{'AP@0.75':>10}{'mAP@[.5:.95]':>15}{'#gt':>6}")
print("-" * 58)
for cname, r in results.items():
    print(f"{cname:<18}{r['AP50']:>9.4f}{r['AP75']:>10.4f}{r['mAP']:>15.4f}{r['n_gt']:>6}")
print("-" * 58)
print(f"{'overall':<18}{'':>9}{'':>10}{overall_map:>15.4f}")

if overall_map > best_map:
    best_map = overall_map
    torch.save({"model": model.state_dict(), "classes": CLASSES,
                "backbone": BACKBONE, "mAP": overall_map}, best_path)
    print(f"\nsaved best model -> {best_path}")

---
## 8. Visualize results

Numbers tell you *how much* the model is wrong; overlays tell you *how*. Look for the
characteristic failure modes: boxes that hug only part of the object (train longer), duplicate
detections (lower the NMS threshold), or confident detections on background clutter (add more
distractors to the Replicator scene).

In [ ]:
@torch.no_grad()
def predict(model, dataset, indices, score_thr=0.5):
    model.eval()
    imgs = [dataset[i][0] for i in indices]
    outs = model([im.to(DEVICE) for im in imgs])
    keep = []
    for out in outs:
        m = out["scores"].cpu().numpy() >= score_thr
        keep.append({
            "boxes": out["boxes"].cpu().numpy()[m],
            "scores": out["scores"].cpu().numpy()[m],
            "labels": out["labels"].cpu().numpy()[m],
        })
    return imgs, keep

def show_predictions(dataset, n=6, score_thr=0.5, seed=RANDOM_SEED + 1):
    idxs = random.Random(seed).sample(range(len(dataset)), min(n, len(dataset)))
    imgs, preds = predict(model, dataset, idxs, score_thr)
    cols = 2
    rows = (len(idxs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(9 * cols, 5 * rows))
    for ax, i, img, pr in zip(np.array(axes).ravel(), idxs, imgs, preds):
        ax.imshow(img.permute(1, 2, 0).numpy())
        _, tgt = dataset[i]
        for box in tgt["boxes"]:                       # ground truth
            x1, y1, x2, y2 = box.tolist()
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           linewidth=2, edgecolor="lime", facecolor="none"))
        for box, score in zip(pr["boxes"], pr["scores"]):   # prediction
            x1, y1, x2, y2 = box
            ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1,
                                           linewidth=2, edgecolor="magenta", facecolor="none"))
            ax.text(x1, y1 - 4, f"{score:.2f}", color="magenta", fontsize=9,
                    bbox=dict(facecolor="black", alpha=0.5, pad=1))
        ax.set_title(f"{dataset.stems[i]} — {len(tgt['boxes'])} GT (green) /"
                     f" {len(pr['boxes'])} pred (magenta)", fontsize=9)
        ax.axis("off")
    for ax in np.array(axes).ravel()[len(idxs):]:
        ax.axis("off")
    fig.suptitle(f"Predictions @ score >= {score_thr}", fontsize=14)
    fig.tight_layout()
    plt.show()

show_predictions(val_ds)

---
## 9. Optional — export for deployment

TorchScript is the least fragile option for a torchvision detector. ONNX export works too but the
post-processing ops (NMS, RoIAlign) need `opset_version >= 11` and a runtime that implements them.

In [ ]:
RUN_EXPORT = False   # flip to True when you are happy with Step 7

if RUN_EXPORT:
    model.eval()
    ts_path = RESULTS_DIR / "detector_scripted.pt"
    torch.jit.save(torch.jit.script(model), ts_path)
    print(f"TorchScript -> {ts_path}  ({ts_path.stat().st_size / 1e6:.0f} MB)")

    onnx_path = RESULTS_DIR / "detector.onnx"
    dummy = torch.randn(1, 3, IMAGE_HEIGHT, IMAGE_WIDTH, device=DEVICE)
    torch.onnx.export(
        model, [dummy], str(onnx_path), opset_version=11,
        input_names=["images"], output_names=["boxes", "labels", "scores"],
        dynamic_axes={"images": {0: "batch", 2: "height", 3: "width"}},
    )
    print(f"ONNX        -> {onnx_path}  ({onnx_path.stat().st_size / 1e6:.0f} MB)")
else:
    print("export skipped — set RUN_EXPORT = True to enable")

---
## Troubleshooting

| Symptom | Cause / fix |
|---|---|
| `CUDA error: no kernel image is available` | torch build has no `sm_` binary for this GPU. Install a newer build (see Step 0). |
| `CUDA out of memory` | Halve `BATCH_SIZE`; or lower `min_size`/`max_size` in Step 3; or set `trainable_layers=1`. |
| Loss becomes `NaN` | LR too high for the batch size. Drop `LEARNING_RATE` 10×. The training loop already skips non-finite batches and clips gradients. |
| mAP stays at 0.0 | `TARGET_CLASS` must match column 0 of the KITTI labels exactly, lowercase. Check the histogram in Step 4.3. |
| `DataLoader worker killed` | Lower `NUM_WORKERS`, or raise shared memory. |
| Boxes drawn but all scores tiny | Undertrained — raise `NUM_EPOCHS`, or unfreeze more of the backbone. |
| Good synthetic AP, poor real-world AP | Sim-to-real gap. Add distractors, lighting, texture and camera-pose randomization in Replicator; strengthen the `ColorJitter` in Step 4.4. |

## Next steps

- Re-generate synthetic data with wider domain randomization and compare mAP — the highest-leverage loop in the whole workflow.
- Mix in a small set of labeled **real** images (even 5–10%) and re-train; this usually closes most of the remaining gap.
- Try a heavier backbone (`resnet50`) once the data pipeline is settled.